# Gradio・HTTPサーバ・cloudflared

制作物に操作画面を付けて、インターネットに公開するための道具を紹介します。

- **Gradio** — Pythonの関数に入力欄と出力欄を付けて、Web画面にするライブラリ。HTMLもCSSも書きません
- **HTTPサーバ** — HTMLを配信したり、リクエストに応じて結果を返したりする土台
- **cloudflared** — 手元で動いているアプリに、外部からアクセスできる公開URLを割り当てる道具

卒業制作では、中間発表や研究発表でデモを見せる場面、友人にテストしてもらう場面で使えます。

In [ ]:
!pip install -q gradio

---
# 基本形

`gr.Interface` に、**関数**・**入力**・**出力**の3つを渡すだけです。画面のレイアウトは Gradio が組み立てます。

In [ ]:
import gradio as gr

def greet(name):
    return f"こんにちは、{name}さん"

gr.Interface(
    fn=greet,
    inputs=gr.Textbox(label="名前"),
    outputs=gr.Textbox(label="結果"),
).launch()

---
# 入力コンポーネント

`inputs` に指定できる主な部品です。関数の引数の順番と、リストに並べた順番が対応します。

| 書き方 | 受け取るもの | 関数に渡る値 |
|---|---|---|
| `gr.Textbox()` | 文字。`lines=5` で複数行 | `str` |
| `gr.Number()` | 数値 | `float` |
| `gr.Slider(0, 100)` | 範囲つきの数値 | `float` |
| `gr.Checkbox()` | オン・オフ | `bool` |
| `gr.Radio([...])` | 選択肢から1つ | `str` |
| `gr.CheckboxGroup([...])` | 選択肢から複数 | `list` |
| `gr.Dropdown([...])` | プルダウン。`multiselect=True` で複数 | `str` / `list` |
| `gr.Image()` | 画像。`type="pil"`／`"numpy"`／`"filepath"` | 画像データ |
| `gr.Audio()` | 音声。`sources=["upload", "microphone"]` | 音声データ |
| `gr.Video()` | 動画 | ファイルパス |
| `gr.File()` | 任意のファイル | ファイル情報 |
| `gr.Dataframe()` | 表。CSVのような入力 | 表データ |
| `gr.ColorPicker()` | 色 | `str`（`#rrggbb`） |
| `gr.Code()` | ソースコード。`language="python"` | `str` |

`label=` で見出し、`value=` で初期値、`info=` で補足説明を付けられます。

In [ ]:
import gradio as gr

def show(text, num, level, agree, plan, tags, color):
    return (
        f"Textbox: {text}\n"
        f"Number: {num}\n"
        f"Slider: {level}\n"
        f"Checkbox: {agree}\n"
        f"Radio: {plan}\n"
        f"CheckboxGroup: {tags}\n"
        f"ColorPicker: {color}"
    )

gr.Interface(
    fn=show,
    inputs=[
        gr.Textbox(label="文字", info="補足説明はここに出ます"),
        gr.Number(label="数値", value=10),
        gr.Slider(0, 100, value=50, label="範囲つきの数値"),
        gr.Checkbox(label="オン・オフ"),
        gr.Radio(["A", "B", "C"], label="1つ選ぶ"),
        gr.CheckboxGroup(["ネットワーク", "セキュリティ", "AI"], label="複数選ぶ"),
        gr.ColorPicker(label="色"),
    ],
    outputs=gr.Textbox(label="受け取った値", lines=8),
    title="入力コンポーネントの例",
).launch()

画像・音声・ファイルも、同じ書き方で扱えます。`type=` で受け取る形式を指定します。`type="pil"` にすると Pillow の画像オブジェクトとして関数に渡るので、そのまま加工できます。

In [ ]:
import gradio as gr

def to_gray(img):
    return img.convert("L")

gr.Interface(
    fn=to_gray,
    inputs=gr.Image(type="pil", label="画像を選ぶ"),
    outputs=gr.Image(type="pil", label="白黒にした画像"),
).launch()

---
# 出力コンポーネント

`outputs` に指定できる主な部品です。複数返すときは、リストに並べた順番と関数の戻り値の順番が対応します。

| 書き方 | 表示されるもの | 関数が返す値 |
|---|---|---|
| `gr.Textbox()` | 文字 | `str` |
| `gr.Label()` | 分類結果と確信度の棒グラフ | `{"ラベル": 確率}` の辞書 |
| `gr.Image()` | 画像 | 画像データ |
| `gr.Audio()` | 音声プレイヤー | 音声データ |
| `gr.Video()` | 動画プレイヤー | ファイルパス |
| `gr.Dataframe()` | 表 | 二次元リスト／DataFrame |
| `gr.Plot()` | グラフ | matplotlib の Figure |
| `gr.JSON()` | JSON の折りたたみ表示 | `dict` / `list` |
| `gr.Gallery()` | 画像の一覧 | 画像のリスト |
| `gr.HTML()` / `gr.Markdown()` | 整形したテキスト | `str` |
| `gr.File()` | ダウンロードできるファイル | ファイルパス |

`gr.Label` は、AIの分類結果を見せるのに向いています。ラベルと確率の辞書を返すと、確信度つきの棒グラフになります。

In [ ]:
import gradio as gr
import matplotlib.pyplot as plt

def analyze(text):
    counts = {
        "alpha": sum(c.isalpha() and c.isascii() for c in text),
        "digit": sum(c.isdigit() for c in text),
        "other": sum(not (c.isalnum() and c.isascii()) for c in text),
    }
    total = max(sum(counts.values()), 1)

    label = {k: v / total for k, v in counts.items()}
    table = [[k, v] for k, v in counts.items()]

    fig, ax = plt.subplots()
    ax.bar(list(counts.keys()), list(counts.values()))
    ax.set_ylabel("count")

    return label, table, fig

gr.Interface(
    fn=analyze,
    inputs=gr.Textbox(label="文字列", value="Network Security 2026"),
    outputs=[
        gr.Label(label="Label 割合"),
        gr.Dataframe(headers=["種類", "個数"], label="Dataframe 表"),
        gr.Plot(label="Plot グラフ"),
    ],
    title="出力コンポーネントの例",
).launch()

グラフの目盛りに日本語を使うと、文字が豆腐（□）になります。matplotlib に日本語フォントを設定するか、目盛りは英数字にしておきます。

---
# 画面を自由に組み立てる（gr.Blocks）

`gr.Interface` は1つの関数に画面を付ける形です。タブで分ける、ボタンを複数置く、部品を横に並べる、といった作りにしたいときは `gr.Blocks` を使います。

- `with gr.Row():` — 横に並べる
- `with gr.Column():` — 縦に並べる
- `with gr.Tab("名前"):` — タブに分ける
- `ボタン.click(関数, 入力, 出力)` — ボタンを押したときの動作を決める

In [ ]:
import gradio as gr

with gr.Blocks(title="Blocks の例") as demo:
    gr.Markdown("## タブと行で組み立てた画面")

    with gr.Tab("計算"):
        with gr.Row():
            a = gr.Number(label="a", value=1)
            b = gr.Number(label="b", value=2)
        out = gr.Textbox(label="a + b")
        gr.Button("計算する").click(lambda x, y: (x or 0) + (y or 0), [a, b], out)

    with gr.Tab("説明"):
        gr.Markdown("Blocks では、部品の配置とボタンの動作を自分で決められます。")

demo.launch()

---
# HTTPサーバ

Gradio が裏でやっているのも、HTTPサーバを立てて画面を配信することです。HTMLを自分で書いて配信したい、他のプログラムから呼び出せるAPIにしたい、という制作物では、HTTPサーバを直接扱います。

## HTTP の中身

やり取りは、リクエストを送るとレスポンスが返る、の一往復です。

- **メソッド** — 何をしたいか。`GET`（取得）、`POST`（送信）など
- **パス** — どこに対してか。`/`、`/api/time` など
- **ステータスコード** — 結果がどうだったか。`200` 成功、`404` 見つからない、`500` サーバ側のエラー
- **ヘッダ** — `Content-Type` などの付随情報
- **ボディ** — 本文のデータ

## 自分でサーバを立てる

Python標準の `http.server` を使います。`BaseHTTPRequestHandler` を継承したクラスに、`do_GET` と `do_POST` を書きます。パスごとに何を返すかを決めるのが、サーバを作るということです。

サーバは待ち受けたまま動き続けるため、ノートブックが固まらないよう別スレッドで起動します。

In [ ]:
import json
import time
import threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

class Handler(BaseHTTPRequestHandler):
    def _send(self, code, body, content_type):
        data = body.encode("utf-8")
        self.send_response(code)                      # ステータスコード
        self.send_header("Content-Type", content_type)
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)                        # ボディを書き込む

    def do_GET(self):
        if self.path == "/":
            self._send(200, "<h1>自分で立てたサーバ</h1>", "text/html; charset=utf-8")
        elif self.path == "/api/time":
            body = json.dumps({"time": time.strftime("%H:%M:%S")})
            self._send(200, body, "application/json; charset=utf-8")
        else:
            self._send(404, json.dumps({"error": "not found"}), "application/json; charset=utf-8")

    def do_POST(self):
        length = int(self.headers.get("Content-Length", 0))
        data = json.loads(self.rfile.read(length) or b"{}")
        body = json.dumps({"received": data}, ensure_ascii=False)
        self._send(200, body, "application/json; charset=utf-8")

    def log_message(self, *args):
        pass                                          # アクセスログを表示しない

PORT = 8000
server = ThreadingHTTPServer(("0.0.0.0", PORT), Handler)
threading.Thread(target=server.serve_forever, daemon=True).start()
print(f"http://localhost:{PORT}/ で起動しました")

## クライアントから呼び出す

リクエストを送る側がクライアントです。`requests` を使うと、辞書をそのまま送って、返事も辞書で受け取れます。

- `requests.get(URL)` — GETで取得する
- `requests.post(URL, json=辞書)` — 辞書をJSONとして送る
- `.status_code` でステータスコード、`.headers` でヘッダ、`.json()` で本文を辞書として取り出す

In [ ]:
import requests

BASE = "http://localhost:8000"

res = requests.get(BASE + "/")
print("ステータス:", res.status_code)
print("Content-Type:", res.headers["Content-Type"])
print("ボディ:", res.text)

print(requests.get(BASE + "/api/time").json())
print(requests.post(BASE + "/", json={"name": "太郎"}).json())
print("存在しないパス:", requests.get(BASE + "/none").status_code)

`curl` でも同じことができます。`-I` を付けるとレスポンスヘッダだけを表示するので、ステータスコードや `Content-Type` の確認に使えます。

In [ ]:
!curl -sI http://localhost:8000/

## ファイルをそのまま配信する

自分でハンドラを書かなくても、ディレクトリの中身を配信するだけなら1行で済みます。作ったHTMLの表示確認に使えます。

```
python3 -m http.server 8000
```

## 大きくなってきたら

パスの数が増える、テンプレートでHTMLを組み立てる、という規模になったら Flask や FastAPI を使います。完成したものを24時間公開するなら、VPS上の Apache などのWebサーバに置きます。

| 道具 | 向いている用途 | 常時稼働 |
|---|---|---|
| `http.server` | 動作確認、小さなAPI、ファイルの配信 | 向かない |
| Flask / FastAPI | パスや処理が増えるWebアプリ、API | 別途サーバが必要 |
| Gradio | 画面付きのデモ、AIモデルの動作見せ | 向かない |
| Apache | 完成した制作物の公開 | 向いている |

---
# 公開する（1）Gradio の共有機能

`launch(share=True)` を付けると、`https://xxxxxxxx.gradio.live` という公開URLが一緒に発行されます。スマートフォンからでも、他の人の端末からでも開けます。

- 有効期限は1週間
- 処理しているのは自分のColabやPCなので、そちらを閉じると動かなくなる
- Gradio のアプリでしか使えない

In [ ]:
import gradio as gr

demo = gr.Interface(fn=lambda s: s[::-1],
                    inputs=gr.Textbox(label="文字列"),
                    outputs=gr.Textbox(label="逆順"))

demo.launch(share=True)

In [ ]:
demo.close()

---
# 公開する（2）cloudflared

cloudflared は、**ポート番号を指定して外に出す道具**です。Gradio でも、さきほどの `http.server` でも、Flask でも、Apache でも、同じやり方で公開できます。アカウント登録は不要です。

```
cloudflared tunnel --url http://localhost:8000
```

`--url` に、公開したいアプリが動いているアドレスを指定するだけです。ポート番号を変えれば、どのサーバでも公開できます。

In [ ]:
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

Colab から使う場合は、cloudflared を動かしたまま次の処理に進む必要があるため、バックグラウンドで起動し、ログから発行されたURLを取り出します。ここでは、さきほど 8000 番で立てた `http.server` を公開します。

In [ ]:
import re
import time
import subprocess

proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
started = time.time()
for line in proc.stdout:
    found = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if found:
        public_url = found.group(0)
        break
    if time.time() - started > 90:
        break

print("公開URL:", public_url)

In [ ]:
proc.terminate()
server.shutdown()
print("トンネルとサーバを停止しました")

---
# 使うときの注意

公開URLは、**インターネットに自分のプログラムを直接さらしている**状態です。リンクを知っている人は誰でもアクセスでき、画面から操作できます。

- 公開する画面に、パスワードや個人情報を入力させない。必要なら `launch(auth=("ユーザ名", "パスワード"))` で認証をかける
- APIキーや接続情報をコードに直接書かない
- 使い終わったら必ず止める。`demo.close()`、`server.shutdown()`、`proc.terminate()`、またはランタイムの終了

## Colab で公開する場合の限界

Colab のランタイムは、放置すると切断されます。切断された時点で公開URLも死ぬため、**発表やテストのためのデモ用**と考えてください。

| やりたいこと | 方法 |
|---|---|
| 発表でデモを見せる、友人に試してもらう | Gradio の共有、または cloudflared |
| 24時間動かし続ける | VPSやクラウドにサーバを立てる |

常時公開が必要なら、教材の「開発環境」にあるVPS・Apacheの手順に進みます。計画書の「公開方法」の欄には、どちらで公開するのかを書きます。